# modol

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
API_key = os.getenv("GOOGLE_API_KEY")
llm_model = "gemini-2.5-flash"

In [3]:
os.environ["LANGCHAIN_PROJECT"] = "Memory store"

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [5]:
model = ChatGoogleGenerativeAI(
                    model=llm_model,
                    temperature=0,
                    timeout=None,
                    max_retries=2)

# Goal

- These long-term memories will be used to create a personalized chatbot that can remember facts about the user.
- It will save memory "in the hot path", as the user is chatting with it.

# store 

In [7]:
import uuid
from langgraph.store.memory import InMemoryStore
in_memory_store = InMemoryStore() 

In [8]:
# Namespace for the memory to save
user_id = "1"
namespace_for_memory = (user_id, "memories")

In [10]:
# Save a memory to namespace as key and value
key = str(uuid.uuid4())

In [11]:
print(key)

a7a244e1-d405-4481-85fb-fe2ae886a2f5


In [12]:
# The value needs to be a dictionary  
value = {"food_preference" : "I like pizza"}

In [13]:
in_memory_store.put(namespace_for_memory, key, value)

In [14]:
print(in_memory_store)

```
{
  ("1", "memories"): {
    "a7a244e1-d405-4481-85fb-fe2ae886a2f5": {
      "food_preference": "I like pizza"
    }
  }
}
```

## search 

In [15]:
memories = in_memory_store.search(namespace_for_memory)
type(memories)

list

In [18]:
memories[0].dict()

{'namespace': ['1', 'memories'],
 'key': 'a7a244e1-d405-4481-85fb-fe2ae886a2f5',
 'value': {'food_preference': 'I like pizza'},
 'created_at': '2026-04-12T09:33:08.941441+00:00',
 'updated_at': '2026-04-12T09:33:08.941444+00:00',
 'score': None}

In [21]:
print(memories[0].key,"\n",memories[0].value)

a7a244e1-d405-4481-85fb-fe2ae886a2f5 
 {'food_preference': 'I like pizza'}


In [24]:
# Get the memory by namespace and key
memory = in_memory_store.get(namespace_for_memory, key)
memory.dict()

{'namespace': ['1', 'memories'],
 'key': 'a7a244e1-d405-4481-85fb-fe2ae886a2f5',
 'value': {'food_preference': 'I like pizza'},
 'created_at': '2026-04-12T09:33:08.941441+00:00',
 'updated_at': '2026-04-12T09:33:08.941444+00:00'}

# Chatbot with long-term memory

has two types of memory
- Short-term (within-thread) memory: Chatbot can persist conversational history and / or allow interruptions in a chat session.
- Long-term (cross-thread) memory: Chatbot can remember information about a specific user across all chat sessions.
- for short-term we use **checkpointer**
- for long-term memory we use **LangGraph Store**
- The chat history will be saved to short-term memory using the checkpointer.
- The chatbot will reflect on the chat history.
- It will then create and save a memory to the LangGraph Store.
- This memory is accessible in future chat sessions to personalize the chatbot's responses.

In [26]:
from IPython.display import Image, display

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.store.base import BaseStore # store 

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.runnables.config import RunnableConfig

In [27]:
# Chatbot instruction
MODEL_SYSTEM_MESSAGE = """You are a helpful assistant with memory that provides information about the user. 
If you have memory for this user, use it to personalize your responses.
Here is the memory (it may be empty): {memory}"""

In [28]:
# Create new memory from the chat history and any existing memory
CREATE_MEMORY_INSTRUCTION = """"You are collecting information about the user to personalize your responses.

CURRENT USER INFORMATION:
{memory}

INSTRUCTIONS:
1. Review the chat history below carefully
2. Identify new information about the user, such as:
   - Personal details (name, location)
   - Preferences (likes, dislikes)
   - Interests and hobbies
   - Past experiences
   - Goals or future plans
3. Merge any new information with existing memory
4. Format the memory as a clear, bulleted list
5. If new information conflicts with existing memory, keep the most recent version

Remember: Only include factual information directly stated by the user. Do not make assumptions or inferences.

Based on the chat history below, please update the user information:"""

## model call 

In [29]:
def call_model(state: MessagesState, config: RunnableConfig, store: BaseStore):

    """Load memory from the store and use it to personalize the chatbot's response."""
    
    # Get the user ID from the config
    user_id = config["configurable"]["user_id"]

    # Retrieve memory from the store
    namespace = ("memory", user_id)
    key = "user_memory"
    existing_memory = store.get(namespace, key)

    # Extract the actual memory content if it exists and add a prefix
    if existing_memory:
        # Value is a dictionary with a memory key
        existing_memory_content = existing_memory.value.get('memory')
    else:
        existing_memory_content = "No existing memory found."

    # Format the memory in the system prompt
    system_msg = MODEL_SYSTEM_MESSAGE.format(memory=existing_memory_content)
    
    # Respond using memory as well as the chat history
    response = model.invoke([SystemMessage(content=system_msg)]+state["messages"])

    return {"messages": response}

In [31]:
def write_memory(state: MessagesState, config: RunnableConfig, store: BaseStore):

    """Reflect on the chat history and save a memory to the store."""
    
    # Get the user ID from the config
    user_id = config["configurable"]["user_id"]

    # Retrieve existing memory from the store
    namespace = ("memory", user_id)
    existing_memory = store.get(namespace, "user_memory")
        
    # Extract the memory
    if existing_memory:
        existing_memory_content = existing_memory.value.get('memory')
    else:
        existing_memory_content = "No existing memory found."

    # Format the memory in the system prompt
    system_msg = CREATE_MEMORY_INSTRUCTION.format(memory=existing_memory_content)
    new_memory = model.invoke([SystemMessage(content=system_msg)]+state['messages'])

    # Overwrite the existing memory in the store 
    key = "user_memory"

    # Write value as a dictionary with a memory key
    store.put(namespace, key, {"memory": new_memory.content})
